In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.getenv(
    "DATA_ROOT",
    r"C:\Users\steph\OneDrive\Desktop\Springboard\Springboard\Capstone\step2\data"
)).expanduser().resolve()

assert DATA_ROOT.exists(), f"{DATA_ROOT} not found"
print("DATA_ROOT =", DATA_ROOT)


DATA_ROOT = C:\Users\steph\OneDrive\Desktop\Springboard\Springboard\Capstone\step2\data


In [ ]:
import contextlib, json, pandas as pd, os
from tqdm import tqdm
from pycocotools.coco import COCO

meta_rows = []

# Stable-Diffusion
sd_csv = DATA_ROOT / "SD" / "custom_prompts_df.csv"
df_sd = (pd.read_csv(sd_csv)
           .rename(columns={"image_file":"image_id","prompt":"caption"}))
df_sd["image_id"] = df_sd["image_id"].str.replace(r"^images[\\/]", "", regex=True)
for _, r in tqdm(df_sd.iterrows(), total=len(df_sd), desc="SD"):
    meta_rows.append(dict(
        id      = f"sd_{_}",
        split   = "train",
        domain  = "sd",
        image_path = str(DATA_ROOT / "SD" / "images" / r.image_id),
        caption = r.caption))

# COCO
ann_json = DATA_ROOT / "coco" / "annotations" / "captions_train2017.json"
with open(os.devnull, "w") as fnull:
    with contextlib.redirect_stdout(fnull):
        coco = COCO(str(ann_json))
for a in tqdm(coco.loadAnns(coco.getAnnIds()), desc="COCO"):
    meta_rows.append(dict(
        id      = f"co_{a['id']}",
        split   = "train",
        domain  = "coco",
        image_path = str(DATA_ROOT / "coco" / "train2017" / f"{int(a['image_id']):012d}.jpg"),
        caption = a["caption"]))

# Flickr-30k
f30_csv = DATA_ROOT / "flickr30k" / "results.csv"

# Load with pipe-separator, strip any stray whitespace in headers
df_f30 = pd.read_csv(f30_csv, sep="|", engine="python")
df_f30.columns = df_f30.columns.str.strip()      
df_f30 = df_f30.rename(columns={"image_name": "image_id",
                                 "comment":    "caption"})

for idx, r in tqdm(df_f30.iterrows(),
                   total=len(df_f30), desc="Flickr"):
    meta_rows.append(dict(
        id         = f"fl_{idx}",                     
        split      = "train",
        domain     = "flickr",
        image_path = str(DATA_ROOT / "flickr30k"
                                   / "flickr30k_images" / r["image_id"]),
        caption    = r["caption"]                    
    ))

meta = pd.DataFrame(meta_rows)
out_pq = Path("../data/metadata.parquet")
out_pq.parent.mkdir(exist_ok=True)
meta.to_parquet(out_pq, index=False)
print(f"Wrote {len(meta):,} rows ➜ {out_pq}")


Flickr: 100%|████████████████████████████████████████████████████████████████| 158915/158915 [00:17<00:00, 8930.93it/s]


Wrote 850,668 rows ➜ ..\data\metadata.parquet
